# Agent SDK Practice Notebook

Use this notebook as a fast, beginner-friendly walkthrough of the `openai_sdk/agent_sdk` examples.

## What you will practice
- Minimal agent run
- Streaming output tokens
- Structured output with `pydantic`
- Function tools
- Multi-turn memory (`SQLiteSession` + `previous_response_id`)
- Orchestration handoffs
- Input guardrails and approval interruptions


## 1. Setup

Before running cells:
1. Ensure `sandbox.yaml` is correctly configured (OCI profile, project, compartment).
2. Install dependencies with `uv sync` (or your project standard install step).
3. `.env` is optional for most Agent SDK examples.

Run script pattern from project root:
- `uv run python -m openai_sdk.agent_sdk.<script_name_without_py>`


In [ ]:
import json

from agents import (
    Agent,
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    Runner,
    SQLiteSession,
    RunContextWrapper,
    TResponseInputItem,
    function_tool,
    input_guardrail,
)
from openai.types.responses import ResponseTextDeltaEvent
from pydantic import BaseModel

from openai_sdk.openai_client_provider import OpenAIClientProvider

MODEL_ID = "openai.gpt-5.2"
print("Step 1/1: Configuring Agents SDK with OCI-backed client...")
OpenAIClientProvider().configure_agents_oci_env()
print("Agents SDK ready.")


## 2. Minimal Agent (`simple_agent.py`)

Start with a single agent + one prompt.


In [ ]:
history_agent = Agent(
    name="History tutor",
    instructions="You answer history questions clearly and concisely.",
    model=MODEL_ID,
)

async def run_once(prompt: str) -> str:
    print(f"Running minimal agent with prompt: {prompt}")
    result = await Runner.run(history_agent, prompt)
    return result.final_output


In [ ]:
await run_once("When did the Roman Empire fall?")


## 3. Streaming Output (`streaming_agent.py`)

This example prints token deltas as they arrive.


In [ ]:
streaming_agent = Agent(
    name="Planet guide",
    instructions="Answer with short facts.",
    model=MODEL_ID,
)

async def stream_once(prompt: str) -> str:
    stream = Runner.run_streamed(streaming_agent, prompt)
    print("Streaming output:")
    async for event in stream.stream_events():
        if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
            print(event.data.delta, end="", flush=True)
    print("\nStreaming complete.")
    return stream.final_output


In [ ]:
await stream_once("Give me three short facts about Saturn.")


## 4. Structured Output (`structured_agent.py`)

Use `output_type` so the answer is validated into a Python object.


In [ ]:
class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

structured_agent = Agent(
    name="Calendar extractor",
    instructions="Extract calendar events from text.",
    model=MODEL_ID,
    output_type=CalendarEvent,
)

print("Running structured-output agent...")
structured_result = await Runner.run(
    structured_agent,
    "Dinner with Priya and Sam on Friday.",
)
print("Parsed result:")
structured_result.final_output


## 5. Function Tool (`use_tool.py`)

Register a Python function and let the model call it when useful.


In [ ]:
@function_tool
def get_history_fun_fact() -> str:
    return "Sharks are older than trees."

tool_agent = Agent(
    name="History tutor with tools",
    instructions="Answer history questions clearly. Use get_history_fun_fact when helpful.",
    model=MODEL_ID,
    tools=[get_history_fun_fact],
)

print("Running tool-enabled agent...")
tool_result = await Runner.run(
    tool_agent,
    "Tell me something surprising about ancient life on Earth.",
)
print("Final response:")
tool_result.final_output


## 6. Multi-turn Memory (`multiturn.py`)

Compare local SQLite memory and server-linked memory.


In [ ]:
tour_agent = Agent(
    name="Tour guide",
    instructions="Answer with compact travel facts. Reply concisely.",
    model=MODEL_ID,
)

# Local memory with SQLiteSession
print("Running local-memory conversation...")
session = SQLiteSession("notebook_conversation")
turn_1 = await Runner.run(tour_agent, "What city is the Golden Gate Bridge in?", session=session)
turn_2 = await Runner.run(tour_agent, "What state is it in?", session=session)

print("Local turn 1:", turn_1.final_output)
print("Local turn 2:", turn_2.final_output)

# Server-linked memory with previous_response_id
print("Running server-linked conversation...")
server_turn_1 = await Runner.run(tour_agent, "What city is the Statue of Liberty in?")
server_turn_2 = await Runner.run(
    tour_agent,
    "What state is it in?",
    previous_response_id=server_turn_1.last_response_id,
)

print("Server turn 1:", server_turn_1.final_output)
print("Server turn 2:", server_turn_2.final_output)


## 7. Orchestration Handoffs (`orchestration.py`)

Route tasks to specialist agents via a triage agent.


In [ ]:
history_specialist = Agent(
    name="History tutor",
    handoff_description="Specialist for history questions.",
    instructions="Answer history questions clearly and concisely.",
    model=MODEL_ID,
)

math_specialist = Agent(
    name="Math tutor",
    handoff_description="Specialist for math questions.",
    instructions="Explain math step by step and include worked examples.",
    model=MODEL_ID,
)

triage_agent = Agent(
    name="Homework triage",
    instructions="Route each homework question to the right specialist.",
    model=MODEL_ID,
    handoffs=[history_specialist, math_specialist],
)

print("Running orchestration handoff example...")
handoff_result = await Runner.run(triage_agent, "Who was the first president of the United States?")
print(handoff_result.final_output)
print("Answered by:", handoff_result.last_agent.name)


## 8. Input Guardrail (`guardail.py`)

This blocks math-homework requests before they reach the main support agent.


In [ ]:
class MathHomeworkOutput(BaseModel):
    is_math_homework: bool
    reasoning: str

guardrail_classifier = Agent(
    name="Homework check",
    instructions="Detect whether the user is asking for math homework help.",
    output_type=MathHomeworkOutput,
    model=MODEL_ID,
)

@input_guardrail
async def math_guardrail(
    ctx: RunContextWrapper[None],
    agent: Agent,
    input: str | list[TResponseInputItem],
) -> GuardrailFunctionOutput:
    result = await Runner.run(guardrail_classifier, input, context=ctx.context)
    print("Guardrail classifier:", result.final_output)
    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=result.final_output.is_math_homework,
    )

support_agent = Agent(
    name="Customer support",
    instructions="Help customers with support questions.",
    input_guardrails=[math_guardrail],
    model=MODEL_ID,
)

print("Running guardrail example...")
try:
    guardrail_result = await Runner.run(support_agent, "Can you solve 2x + 3 = 11 for me?")
    print("Guardrail passed:", guardrail_result.final_output)
except InputGuardrailTripwireTriggered:
    print("Guardrail blocked the request.")


## 9. Approval Interruptions (`guardail_approval.py`)

The notebook version below previews interruption data and programmatic approval.


In [ ]:
@function_tool(needs_approval=True)
async def cancel_order(order_id: int) -> str:
    return f"Cancelled order {order_id}"

approval_agent = Agent(
    name="Support agent",
    instructions="Handle support requests and ask for approval when needed.",
    tools=[cancel_order],
    model=MODEL_ID,
)

print("Running approval interruption example...")
approval_result = await Runner.run(approval_agent, "Cancel order 123.")

if approval_result.interruptions:
    print("Interruptions found:")
    for interruption in approval_result.interruptions:
        print("-", interruption.qualified_name or interruption.name)
        try:
            print(json.dumps(json.loads(interruption.arguments or "{}"), indent=2))
        except json.JSONDecodeError:
            print(interruption.arguments)

    # Notebook demo: auto-approve all interruptions.
    state = approval_result.to_state()
    for interruption in approval_result.interruptions:
        state.approve(interruption)

    approval_result = await Runner.run(approval_agent, state)

print("Final response:")
approval_result.final_output


## 10. Try These Experiments

1. Change only one variable at a time (prompt, model, or instructions) and compare behavior.
2. Add one new tool (for example `lookup_shipping_status`) and teach the agent when to use it.
3. Expand `CalendarEvent` with optional fields and observe parsing reliability.
4. Change guardrail classifier instructions and evaluate false positives/negatives.
5. Add a third specialist to orchestration and test routing quality.
6. Optional: run `voice_agent.py` from project root:
   - `uv run python -m openai_sdk.agent_sdk.voice_agent`

## Discussion prompts
- Which pattern felt most predictable: structured output, tools, or guardrails?
- When should you prefer local memory (`SQLiteSession`) vs server-linked memory?
- What risks should always require human approval in your use case?
